In [1]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create and set the working directory
BASE_PATH = '/content/drive/MyDrive/voice-lab-data/'
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)
print(f"Working directory set to: {os.getcwd()}")

Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/voice-lab-data


In [ ]:
%%writefile xtts_pipeline_utils.py

import json
import time
import pickle
import re
import shutil
import subprocess
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import librosa
from tqdm.auto import tqdm
import requests

try:
    from TTS.api import TTS
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
    TTS_AVAILABLE = True
except ImportError:
    TTS_AVAILABLE = False

try:
    from speechbrain.pretrained import SpeakerRecognition
    SPEECHBRAIN_AVAILABLE = True
except ImportError:
    SPEECHBRAIN_AVAILABLE = False

# --- CONFIGURATION ---
@dataclass
class OptimizedXTTSConfig:
    """Optimized configuration for XTTS."""
    data_path: Path
    model_path: Path
    output_path: Path
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    mixed_precision: bool = True
    sample_rate: int = 22050
    hop_length: int = 256
    win_length: int = 1024
    n_fft: int = 1024
    n_mels: int = 80
    mel_fmin: int = 0
    mel_fmax: Optional[int] = None
    batch_size: int = 2 if torch.cuda.is_available() else 1
    eval_batch_size: int = 1
    num_loader_workers: int = 2
    epochs: int = 1000
    save_step: int = 5000
    print_step: int = 100
    lr: float = 5e-6
    weight_decay: float = 1e-6
    grad_clip: float = 1.0
    languages: List[str] = field(default_factory=lambda: ["en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru", "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko", "tw"])
    min_audio_len: float = 1.0
    max_audio_len: float = 15.0
    min_text_len: int = 5
    max_text_len: int = 500
    datasets_to_scan: List[Dict] = field(default_factory=list)

    def __post_init__(self):
        self.output_path.mkdir(parents=True, exist_ok=True)
        self.model_path.mkdir(parents=True, exist_ok=True)
        if torch.cuda.is_available():
            gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            if gpu_memory_gb < 12: self.batch_size = 1
            elif gpu_memory_gb < 24: self.batch_size = 2
            else: self.batch_size = 4

# --- DATA LOADER (FINAL CUSTOM-BUILT VERSION) ---
class ComprehensiveTTSDataLoader:
    """Scans and processes a predefined set of raw directories defined in the config."""
    def __init__(self, config: OptimizedXTTSConfig, paths: Dict[str, Path]):
        self.config = config
        self.paths = paths
        self.speaker_stats = None

    def create_unified_dataset(self) -> pd.DataFrame:
        print("🔄 Creating unified TTS dataset using custom scanners for your data...")
        all_pairs = []
        datasets_to_scan = self.config.datasets_to_scan

        for dataset in datasets_to_scan:
            pairs = []
            print(f"--- Scanning dataset: {dataset['name']} ---")
            if dataset['type'] == 'bibletts':
                pairs = self._scan_bibletts_structure(dataset['path'])
            elif dataset['type'] == 'financial_inclusion':
                pairs = self._scan_financial_inclusion_structure(dataset['path'])
            elif dataset['type'] == 'arrow_sharded_audio':
                pairs = self._scan_arrow_sharded_audio(dataset['path'], dataset['name'])

            if pairs:
                print(f"   Found {len(pairs)} samples in {dataset['name']}")
                all_pairs.extend(pairs)

        if not all_pairs:
            print("   ⚠️ No audio/text pairs found. Please double-check paths in your config. Falling back to dummy data.")
            return self._create_dummy_dataset()

        unified_df = pd.DataFrame(all_pairs)
        unified_df = self._add_audio_metadata(unified_df)
        unified_df = self._quality_filter(unified_df)
        self._generate_speaker_stats(unified_df)
        return unified_df

    def _scan_arrow_sharded_audio(self, dataset_path: Path, dataset_name: str) -> List[Dict]:
        """Scans directories with sharded Arrow files containing embedded audio."""
        pairs = []
        temp_audio_dir = self.paths['base'] / 'temp_audio' / dataset_name
        temp_audio_dir.mkdir(parents=True, exist_ok=True)
        arrow_files = sorted(list(dataset_path.glob('data-*.arrow')))
        if not arrow_files:
            print(f"   ⚠️ No .arrow files found in {dataset_path}")
            return pairs
        try:
            df = pd.concat([pd.read_feather(f) for f in arrow_files], ignore_index=True)
            # Find relevant columns (Updated with correct names)
            text_col = next((c for c in ['transcription', 'transcript', 'text', 'sentence'] if c in df.columns), None)
            audio_col = next((c for c in ['audio_path', 'audio', 'speech'] if c in df.columns), None)
            speaker_col = next((c for c in ['speaker_id', 'client_id'] if c in df.columns), 'client_id')

            if not (text_col and audio_col):
                print(f"   ❌ Missing required 'text' or 'audio' columns in Arrow files at {dataset_path}")
                return pairs

            for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {dataset_name}", leave=False):
                audio_data, text = row.get(audio_col), row.get(text_col)
                if not (audio_data and isinstance(text, str)): continue

                speaker_id = row.get(speaker_col, 'unknown_speaker')
                audio_array = np.array(audio_data['array'], dtype=np.float32)
                sampling_rate = audio_data['sampling_rate']
                temp_filename = f"{dataset_name}_{speaker_id}_{index}.wav"
                temp_filepath = temp_audio_dir / temp_filename
                sf.write(temp_filepath, audio_array, sampling_rate)
                pairs.append({'audio_path': str(temp_filepath), 'text': text, 'speaker_id': f"{dataset_name}_{speaker_id}", 'language': 'tw', 'dataset': dataset_name})
        except Exception as e:
            print(f"   ❌ Error processing Arrow files in {dataset_path}: {e}")
        return pairs

    def _scan_bibletts_structure(self, bibletts_path: Path) -> List[Dict]:
        """Corrected scanner for bibletts, looking for co-located .txt files."""
        pairs = []
        audio_files = [p for ext in ['.wav', '.mp3', '.flac', '.ogg'] for p in bibletts_path.glob(f'**/*{ext}')]

        for audio_file in tqdm(audio_files, desc="Scanning bibletts", leave=False):
            if '__MACOSX' in str(audio_file): continue

            text_file = audio_file.with_suffix('.txt')
            if text_file.exists():
                try:
                    text = text_file.read_text().strip()
                    if text:
                        parts = audio_file.parts
                        dialect_name = parts[parts.index('bibletts') + 1]
                        speaker_id = f'bibletts_{dialect_name}_{audio_file.parent.name}'
                        pairs.append({'audio_path': str(audio_file), 'text': text, 'speaker_id': speaker_id, 'language': 'tw', 'dataset': 'bibletts'})
                except Exception: continue
        return pairs

    def _scan_financial_inclusion_structure(self, financial_path: Path) -> List[Dict]:
        """Scanner that finds the data.csv manifest for financial_inclusion data."""
        pairs = []
        audio_files = [p for ext in ['.wav', '.mp3', '.ogg', '.flac'] for p in financial_path.glob(f'**/audios/*{ext}')]
        manifest_cache = {}

        for audio_file in tqdm(audio_files, desc="Scanning financial_inclusion", leave=False):
            if '__MACOSX' in str(audio_file): continue

            manifest_path = audio_file.parent.parent / 'data.csv'
            if not manifest_path.exists(): continue

            if str(manifest_path) not in manifest_cache:
                try:
                    df = pd.read_csv(manifest_path, on_bad_lines='skip')
                    # Find the audio and text columns, whatever they may be named
                    path_col = next((c for c in df.columns if 'path' in c.lower() or 'audio' in c.lower() or 'filename' in c.lower()), None)
                    text_col = next((c for c in df.columns if 'text' in c.lower() or 'transcript' in c.lower()), None)
                    if not (path_col and text_col):
                        manifest_cache[str(manifest_path)] = {}
                        continue
                    manifest_cache[str(manifest_path)] = df.set_index(path_col)[text_col].to_dict()
                except Exception:
                    manifest_cache[str(manifest_path)] = {}

            transcripts = manifest_cache.get(str(manifest_path), {})
            # Key might be just the filename, or relative path like 'audios/filename.ogg'
            transcript = transcripts.get(audio_file.name) or transcripts.get(f"audios/{audio_file.name}")
            if transcript and isinstance(transcript, str):
                speaker_id = f"fisd_{audio_file.parent.parent.name}"
                pairs.append({'audio_path': str(audio_file), 'text': transcript, 'speaker_id': speaker_id, 'language': 'tw', 'dataset': 'financial_inclusion'})
        return pairs

    def _add_audio_metadata(self, df: pd.DataFrame) -> pd.DataFrame:
        print("   Adding audio metadata...")
        metadata = []
        for path in tqdm(df['audio_path'], desc="Reading audio metadata"):
            try:
                info = sf.info(path)
                metadata.append({'duration': info.duration, 'sample_rate': info.samplerate, 'audio_exists': True})
            except Exception:
                metadata.append({'duration': 0.0, 'sample_rate': 0, 'audio_exists': False})
        meta_df = pd.DataFrame(metadata)
        df = pd.concat([df.reset_index(drop=True), meta_df], axis=1)
        df['text_length'] = df['text'].str.len()
        return df

    def _quality_filter(self, df: pd.DataFrame) -> pd.DataFrame:
        print("   Applying quality filters...")
        initial_count = len(df)
        df_filtered = df[(df['audio_exists']) & (df['duration'] >= self.config.min_audio_len) & (df['duration'] <= self.config.max_audio_len) & (df['text_length'] >= self.config.min_text_len) & (df['text_length'] <= self.config.max_text_len)].copy()
        retained_percent = (len(df_filtered) / initial_count * 100) if initial_count > 0 else 0
        print(f"   Filtered: {initial_count} -> {len(df_filtered)} samples ({retained_percent:.1f}% retained)")
        return df_filtered

    def _generate_speaker_stats(self, df: pd.DataFrame):
        print("   Generating speaker statistics...")
        if 'speaker_id' in df.columns and not df.empty:
            self.speaker_stats = df.groupby('speaker_id').agg(sample_count=('audio_path', 'count'), total_duration=('duration', 'sum')).round(2)
            print(f"   📊 Found {len(self.speaker_stats)} unique speakers.")
        return self.speaker_stats

    def _create_dummy_dataset(self) -> pd.DataFrame:
        print("   Creating dummy TTS dataset for testing...")
        dummy_dir = self.paths['data'] / 'tts_ready' / 'dummy_tts'
        dummy_dir.mkdir(parents=True, exist_ok=True)
        samples = [("Wo ho te sɛn? Me ho yɛ.", "tw", "speaker_001"), ("This is a test.", "en", "speaker_003")]
        dummy_data = []
        for i, (text, lang, speaker) in enumerate(samples):
            audio_file = dummy_dir / f"{speaker}_{i:03d}.wav"
            if not audio_file.exists():
                dummy_audio = np.random.normal(0, 0.1, int(22050 * 5.0)).astype(np.float32)
                sf.write(audio_file, dummy_audio, 22050)
            info = sf.info(str(audio_file))
            dummy_data.append({'audio_path': str(audio_file), 'text': text, 'speaker_id': speaker, 'language': lang, 'duration': info.duration, 'sample_rate': info.samplerate, 'audio_exists': True, 'text_length': len(text)})
        df = pd.DataFrame(dummy_data)
        self._generate_speaker_stats(df)
        return df

# --- REAL XTTS TRAINER WITH ACTUAL TRAINING ---
class AdvancedXTTSTrainer:
    def __init__(self, config: OptimizedXTTSConfig, dataset: pd.DataFrame):
        self.config = config
        self.dataset = dataset
        self.speaker_encoder = None
        self.training_output_path = self.config.output_path / "training_output"
        self.training_output_path.mkdir(parents=True, exist_ok=True)
        self._setup_speaker_recognition()

    def _setup_speaker_recognition(self):
        print("\n🎯 Setting up speaker recognition...")
        if not SPEECHBRAIN_AVAILABLE:
            print("   ⚠️ SpeechBrain not found. Using fallback for embeddings.")
            return
        try:
            self.speaker_encoder = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=str(self.config.model_path / "speaker_encoder"))
            print("   ✅ SpeechBrain speaker encoder loaded")
        except Exception as e:
            print(f"   ⚠️ Could not load SpeechBrain: {e}. Using fallback.")
            self.speaker_encoder = None

    def generate_speaker_embeddings(self) -> Dict[str, np.ndarray]:
        print("\n🔊 Generating speaker embeddings...")
        speaker_embeddings = {}
        representative_samples = self.dataset.groupby('speaker_id').first().reset_index()
        for _, row in tqdm(representative_samples.iterrows(), total=len(representative_samples), desc="Speaker embeddings"):
            speaker_id, audio_path = row['speaker_id'], row['audio_path']
            if self.speaker_encoder:
                try:
                    embedding = self.speaker_encoder.encode_file(audio_path)
                    speaker_embeddings[speaker_id] = embedding.squeeze().cpu().numpy()
                except Exception as e:
                    speaker_embeddings[speaker_id] = np.random.rand(1, 192).astype(np.float32)
            else:
                speaker_embeddings[speaker_id] = np.random.rand(1, 192).astype(np.float32)
        print(f"   ✅ Generated embeddings for {len(speaker_embeddings)} speakers")
        with open(self.config.output_path / "speaker_embeddings.pkl", 'wb') as f:
            pickle.dump(speaker_embeddings, f)
        print(f"   📁 Embeddings saved")
        return speaker_embeddings

    def prepare_coqui_dataset(self) -> Path:
        print("\n📄 Preparing Coqui TTS dataset...")
        dataset_dir = self.config.data_path / "coqui_xtts_dataset"
        audio_dir = dataset_dir / "wavs"
        audio_dir.mkdir(parents=True, exist_ok=True)
        metadata_lines = []
        for _, row in tqdm(self.dataset.iterrows(), total=len(self.dataset), desc="Processing audio for Coqui"):
            try:
                # Resample and save audio file to the target format
                audio, sr = librosa.load(row['audio_path'], sr=self.config.sample_rate)
                text = re.sub(r'[^\w\s.,?!]', '', row['text']).strip()
                # Create a unique filename to avoid collisions
                output_filename = f"{row['dataset']}_{row['speaker_id']}_{Path(row['audio_path']).stem}.wav"
                output_path = audio_dir / output_filename
                sf.write(output_path, audio, self.config.sample_rate)
                # Metadata line for Coqui format: relative_path|text|speaker_name
                metadata_lines.append(f"wavs/{output_filename}|{text}|{row['speaker_id']}")
            except Exception as e:
                continue
        self._create_train_val_splits(dataset_dir, metadata_lines)
        print(f"   ✅ Processed {len(metadata_lines)} samples for Coqui format.")
        print(f"   📁 Dataset saved to: {dataset_dir}")
        return dataset_dir

    def _create_train_val_splits(self, dataset_dir: Path, metadata: List[str]):
        print("   Creating train/validation splits...")
        np.random.shuffle(metadata)
        split_idx = int(len(metadata) * 0.95) # 95/5 split
        train_lines, val_lines = metadata[:split_idx], metadata[split_idx:]
        with open(dataset_dir / "metadata_train.csv", 'w', encoding='utf-8') as f:
            f.write('\n'.join(train_lines))
        with open(dataset_dir / "metadata_val.csv", 'w', encoding='utf-8') as f:
            f.write('\n'.join(val_lines))
        print(f"   Train samples: {len(train_lines)}, Validation samples: {len(val_lines)}")

    def download_base_model(self):
        """Download the base XTTS v2 model for fine-tuning."""
        print("\n📥 Downloading base XTTS v2 model...")
        try:
            # Download base model using TTS API
            tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())
            base_model_path = self.training_output_path / "base_model"
            base_model_path.mkdir(parents=True, exist_ok=True)
            
            # Save the base model configuration and weights
            model_path = tts.model_path
            config_path = tts.config_path
            
            if model_path and config_path:
                shutil.copy2(config_path, base_model_path / "config.json")
                if Path(model_path).is_file():
                    shutil.copy2(model_path, base_model_path / "model.pth")
                elif Path(model_path).is_dir():
                    shutil.copytree(model_path, base_model_path / "model", dirs_exist_ok=True)
                
                print(f"   ✅ Base model saved to: {base_model_path}")
                return base_model_path
        except Exception as e:
            print(f"   ❌ Error downloading base model: {e}")
            return None

    def train_model(self, dataset_path: Path) -> bool:
        """Actually execute the XTTS training process."""
        print("\n🚀 Starting XTTS v2 model training...")
        
        # Download base model first
        base_model_path = self.download_base_model()
        if not base_model_path:
            print("   ❌ Failed to download base model. Cannot proceed with training.")
            return False
            
        try:
            # Create training configuration
            config_path = self._create_training_config(dataset_path, base_model_path)
            
            # Execute training command
            cmd = [
                "python", "-m", "TTS.bin.train_tts",
                "--config_path", str(config_path),
                "--restore_path", str(base_model_path / "model.pth"),
            ]
            
            print(f"   Executing: {' '.join(cmd)}")
            print("   📊 Training started - this will take several hours...")
            
            # Start training process
            process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                universal_newlines=True,
                cwd=str(self.training_output_path)
            )
            
            # Monitor training output
            for line in process.stdout:
                line = line.strip()
                if line:
                    # Print important training metrics
                    if any(keyword in line.lower() for keyword in ['epoch', 'loss', 'step', 'eval']):
                        print(f"   {line}")
                        
            process.wait()
            
            if process.returncode == 0:
                print("   ✅ Training completed successfully!")
                self._save_final_model()
                return True
            else:
                print(f"   ❌ Training failed with return code: {process.returncode}")
                return False
                
        except Exception as e:
            print(f"   ❌ Error during training: {e}")
            import traceback
            traceback.print_exc()
            return False

    def _create_training_config(self, dataset_path: Path, base_model_path: Path) -> Path:
        """Create training configuration file."""
        print("   Creating training configuration...")
        
        config_dict = {
            "model": "xtts",
            "run_name": "xtts_fine_tune",
            "dataset": "coqui_ai_dataset", 
            "dataset_config": {
                "formatter": "coqui_ai_dataset",
                "dataset_name": "custom_dataset",
                "path": str(dataset_path),
                "meta_file_train": "metadata_train.csv",
                "meta_file_val": "metadata_val.csv",
                "language": "tw"
            },
            "audio": {
                "sample_rate": self.config.sample_rate,
                "resample": True
            },
            "run_eval": True,
            "test_delay_epochs": 1,
            "epochs": self.config.epochs,
            "text_cleaner": "multilingual_cleaners",
            "use_phonemes": True,
            "phoneme_language": "tw",
            "phoneme_cache_path": str(self.training_output_path / "phoneme_cache"),
            "print_step": self.config.print_step,
            "plot_step": 100,
            "model_param": {
                "args": {
                    "gpt_batch_size": self.config.batch_size,
                    "enable_redaction": False,
                    "kv_cache": True,
                    "gpt_checkpoint": True,
                    "max_conditioning_length": 132300,
                    "max_wav_length": 255995,
                    "max_text_length": 200,
                    "mel_norm_file": str(base_model_path / "mel_stats.pth"),
                    "dvae_checkpoint": str(base_model_path / "dvae.pth"),
                    "xtts_checkpoint": str(base_model_path / "model.pth"),
                    "tokenizer_file": str(base_model_path / "vocab.json"),
                    "gpt_num_audio_tokens": 1024,
                    "gpt_start_audio_token": 1024,
                    "gpt_stop_audio_token": 1025,
                    "gpt_use_masking_gt_prompt_approach": True,
                    "gpt_use_perceiver_resampler": True,
                }
            },
            "lr": self.config.lr,
            "optimizer": "AdamW",
            "optimizer_params": {
                "betas": [0.9, 0.96],
                "eps": 1e-8,
                "weight_decay": self.config.weight_decay
            },
            "lr_scheduler": "MultiStepLR",
            "lr_scheduler_params": {
                "milestones": [50000, 150000, 300000],
                "gamma": 0.5,
                "last_epoch": -1
            },
            "test_sentences_file": "",
            "output_path": str(self.training_output_path),
            "save_step": self.config.save_step,
            "save_n_checkpoints": 5,
            "save_checkpoints": True,
            "target_loss": "loss",
            "print_eval": False,
            "use_cuda": torch.cuda.is_available()
        }
        
        config_path = self.training_output_path / "config.json"
        with open(config_path, 'w') as f:
            json.dump(config_dict, f, indent=2)
            
        print(f"   ✅ Training config saved to: {config_path}")
        return config_path

    def _save_final_model(self):
        """Save the final trained model."""
        print("   💾 Saving final trained model...")
        final_model_path = self.config.output_path / "final_model"
        final_model_path.mkdir(parents=True, exist_ok=True)
        
        # Find the best checkpoint
        checkpoints = list(self.training_output_path.glob("checkpoint_*.pth"))
        if checkpoints:
            best_checkpoint = max(checkpoints, key=lambda x: x.stat().st_mtime)
            shutil.copy2(best_checkpoint, final_model_path / "model.pth")
            
            # Copy config and other necessary files
            config_file = self.training_output_path / "config.json"
            if config_file.exists():
                shutil.copy2(config_file, final_model_path / "config.json")
                
            print(f"   ✅ Final model saved to: {final_model_path}")
        else:
            print("   ⚠️ No checkpoints found to save")

# --- REAL XTTS INFERENCE ---
class OptimizedXTTSInference:
    def __init__(self, config: OptimizedXTTSConfig):
        self.config = config
        self.model = None
        self.trained_model_path = self.config.output_path / "final_model"

    def load_model(self, use_trained_model: bool = True):
        print(f"\n🤖 Loading XTTS model...")
        if not TTS_AVAILABLE:
            print("   ⚠️ TTS not available")
            return False
            
        try:
            # Try to load trained model first if it exists
            if use_trained_model and (self.trained_model_path / "model.pth").exists():
                print("   🎯 Loading fine-tuned model...")
                config_path = self.trained_model_path / "config.json"
                model_path = self.trained_model_path / "model.pth"
                
                if config_path.exists():
                    # Load custom trained model
                    with open(config_path, 'r') as f:
                        config_dict = json.load(f)
                    
                    config = XttsConfig()
                    config.load_json(str(config_path))
                    
                    self.model = Xtts(config)
                    self.model.load_checkpoint(config, checkpoint_path=str(model_path))
                    
                    if torch.cuda.is_available():
                        self.model.cuda()
                        
                    print(f"   ✅ Fine-tuned model loaded from: {self.trained_model_path}")
                else:
                    raise FileNotFoundError("Config file not found")
            else:
                # Fallback to base model
                print("   📥 Loading base XTTS v2 model...")
                self.model = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())
                print(f"   ✅ Base model loaded successfully.")
            return True
            
        except Exception as e:
            print(f"   ❌ Error loading model: {e}")
            return False

    def test_voice_cloning(self, reference_audios: List[str]):
        print("\n🧪 Testing voice cloning capabilities...")
        if not reference_audios or not self.model: return
        
        test_phrases = [("Wo ho te sɛn? Me ho yɛ.", "tw"), ("This is a test of English voice cloning.", "en")]
        for text, lang in test_phrases:
            ref_audio = np.random.choice(reference_audios)
            output_path = self.config.output_path / f"cloned_{lang}_{int(time.time())}.wav"
            print(f"   Cloning '{text[:30]}...' using {Path(ref_audio).name}")
            
            try:
                if hasattr(self.model, 'tts_to_file'):
                    self.model.tts_to_file(text=text, speaker_wav=ref_audio, language=lang, file_path=str(output_path))
                else:
                    # For custom model
                    outputs = self.model.synthesize(text, self.model.config, speaker_wav=ref_audio, language=lang)
                    sf.write(str(output_path), outputs["wav"], self.config.sample_rate)
                    
                print(f"   ✅ Voice cloned to: {output_path}")
            except Exception as e:
                print(f"   ❌ Error during synthesis: {e}")

def real_training_pipeline(trainer: AdvancedXTTSTrainer) -> Tuple[Optional[AdvancedXTTSTrainer], Optional[Path]]:
    """Complete training pipeline that actually trains the model."""
    print("\n🚀 Starting REAL XTTS training pipeline...")
    if not trainer:
        print("   ❌ No trainer provided")
        return None, None
        
    try:
        # Step 1: Generate speaker embeddings
        trainer.generate_speaker_embeddings()
        
        # Step 2: Prepare dataset
        dataset_path = trainer.prepare_coqui_dataset()
        
        # Step 3: Actually train the model
        training_success = trainer.train_model(dataset_path)
        
        if training_success:
            print("\n🎉 REAL training pipeline completed successfully!")
            return trainer, dataset_path
        else:
            print("\n❌ Training failed")
            return None, None
            
    except Exception as e:
        print(f"   ❌ Error in training pipeline: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def benchmark_model_performance(inference_engine: OptimizedXTTSInference, dataset: pd.DataFrame):
    print(f"\n⚡ Benchmarking XTTS model performance...")
    if not inference_engine or not inference_engine.model or dataset.empty:
        print("   ⚠️ Inference engine not loaded or dataset is empty.")
        return
        
    test_cases = [{"name": "Short Twi", "text": "Me ho yɛ.", "lang": "tw", "exp_dur": 1.5}, {"name": "English", "text": "This is a test.", "lang": "en", "exp_dur": 3.0}]
    ref_audio = dataset.iloc[0]['audio_path']
    results = []
    
    for case in test_cases:
        times = []
        print(f"\n   Testing: {case['name']}")
        for i in range(3):
            start_time = time.time()
            output_path = inference_engine.config.output_path / f"benchmark_{i}.wav"
            
            try:
                if hasattr(inference_engine.model, 'tts_to_file'):
                    inference_engine.model.tts_to_file(text=case['text'], speaker_wav=ref_audio, language=case['lang'], file_path=str(output_path))
                else:
                    outputs = inference_engine.model.synthesize(case['text'], inference_engine.model.config, speaker_wav=ref_audio, language=case['lang'])
                    sf.write(str(output_path), outputs["wav"], inference_engine.config.sample_rate)
                    
                elapsed = time.time() - start_time
                times.append(elapsed)
                actual_duration = sf.info(str(output_path)).duration
                print(f"     Run {i+1}: {elapsed:.2f}s (audio: {actual_duration:.2f}s)")
            except Exception as e:
                print(f"     Run {i+1}: Failed - {e}")
                times.append(float('inf'))
                
        avg_time = np.mean([t for t in times if t != float('inf')])
        rtf = avg_time / case['exp_dur'] if case['exp_dur'] > 0 and avg_time != float('inf') else float('inf')
        results.append({'test_case': case['name'], 'avg_time': avg_time, 'rtf': rtf})
        print(f"     ✅ Average: {avg_time:.2f}s | RTF: {rtf:.2f}x")
        
    with open(inference_engine.config.output_path / "performance_benchmark.json", 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n📁 Detailed results saved")

In [3]:
# Install essential system libraries for audio processing and phonemization
print("📦 Installing system dependencies...")
!apt-get update && apt-get install -y libsndfile1-dev espeak-ng

# Create the requirements.txt file
print("\n📝 Creating requirements.txt file...")


📦 Installing system dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,006 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cli.github.com/packages stable/main amd64 Packages [346 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,798 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:13 https:/

In [4]:
%%writefile requirements.txt
torch>=2.0.0
torchaudio>=2.0.0
speechbrain>=0.5.0
transformers>=4.30.0
datasets>=2.12.0
librosa>=0.10.0
soundfile>=0.12.0
scipy>=1.10.0
numpy>=1.24.0
pandas>=2.0.0
tqdm>=4.65.0
pyarrow
torchcodec

# Install TTS from the dev branch of its GitHub repository for better compatibility
print("\n🚀 Installing TTS from GitHub...")
!pip install -q "TTS @ git+https://github.com/coqui-ai/TTS@dev"

# Install all other Python packages
print("\n🚀 Installing other dependencies from requirements.txt...")
!pip install -q -r requirements.txt

print("\n\n✅ All installations completed successfully!")
print("   ‼️ PLEASE RESTART YOUR RUNTIME NOW (Runtime > Restart session) BEFORE PROCEEDING. ‼️")

Overwriting requirements.txt


In [ ]:
import os
import torch
import pandas as pd
from pathlib import Path

# Reload the utility script after installation/restart
import xtts_pipeline_utils
from xtts_pipeline_utils import (
    OptimizedXTTSConfig,
    ComprehensiveTTSDataLoader,
    AdvancedXTTSTrainer,
    OptimizedXTTSInference,
    real_training_pipeline,  # Updated to use real training
    benchmark_model_performance,
)

# Define the directory structure
BASE_PATH = Path(os.getcwd())
PATHS = {
    'base': BASE_PATH,
    'data': BASE_PATH / 'data',
    'models': BASE_PATH / 'models',
    'xtts_models': BASE_PATH / 'models' / 'xtts',
    'output': BASE_PATH / 'output',
    'tts_ready': BASE_PATH / 'data' / 'tts_ready', # Added tts_ready path
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print("✅ Environment and paths configured successfully!")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# (In Cell 5 of your notebook)

# 1. Initialize the configuration
base_scan_path = PATHS['base'] / 'raw'

xtts_config = OptimizedXTTSConfig(
    data_path=PATHS['data'],
    model_path=PATHS['xtts_models'],
    output_path=PATHS['output'],
    # --- Define ALL the datasets you want to scan here ---
    datasets_to_scan=[
        {
            "name": "bibletts",
            "type": "bibletts",
            "path": base_scan_path / 'bibletts'
        },
        {
            "name": "financial_inclusion",
            "type": "financial_inclusion",
            "path": base_scan_path / 'financial_inclusion'
        },
        # --- Add ALL your Arrow datasets here ---
        { "name": "akuapem_multi_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/akuapem_multispeaker_audio_transcribed/train" },
        { "name": "akuapem_multi_test", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/akuapem_multispeaker_audio_transcribed/test" },
        { "name": "fante_multi_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/fante_multispeaker_audio_transcribed/train" },
        { "name": "fante_multi_test", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/fante_multispeaker_audio_transcribed/test" },
        { "name": "ga_multi_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/ga_multispeaker_audio_transcribed/train" },
        { "name": "ga_multi_test", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/ga_multispeaker_audio_transcribed/test" },
        { "name": "twi_multi_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/twi_multispeaker_audio_transcribed/train" },
        { "name": "twi_multi_test", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/twi_multispeaker_audio_transcribed/test" },
        { "name": "twi_emotions_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/twi-emotions-corpus/train" },
        { "name": "twi_parallel_train", "type": "arrow_sharded_audio", "path": base_scan_path / "michsethowusu/twi-speech-text-parallel-multispeaker/train" },
    ]
)
print(f"⚙️ Optimized XTTS Configuration Initialized.")

# 2. Load the dataset
data_loader = ComprehensiveTTSDataLoader(xtts_config, PATHS)
tts_dataset = data_loader.create_unified_dataset()

print("\n📊 Dataset Summary:")
if not tts_dataset.empty:
    display(tts_dataset.head())
    print(f"\nTotal samples found: {len(tts_dataset)}")

⚙️ Optimized XTTS Configuration Initialized.
🔄 Creating unified TTS dataset using custom scanners for your data...
--- Scanning dataset: bibletts ---


Scanning bibletts:   0%|          | 0/46841 [00:00<?, ?it/s]

In [ ]:
# 3. Initialize the trainer
if not tts_dataset.empty:
    xtts_trainer = AdvancedXTTSTrainer(xtts_config, tts_dataset)
    print("\n✅ Advanced XTTS trainer initialized!")
else:
    xtts_trainer = None
    print("\n⚠️ No TTS data available for training.")

# 4. Initialize the inference engine
xtts_inference = OptimizedXTTSInference(xtts_config)
print("✅ XTTS inference engine initialized!")

In [ ]:
# 5. Run the REAL XTTS v2 training pipeline
print("🚀 Starting REAL XTTS v2 Model Training Pipeline")
print("=" * 60)

trainer, dataset_path = real_training_pipeline(xtts_trainer)

if trainer and dataset_path:
    print("\n✅ REAL XTTS v2 training pipeline completed successfully!")
    
    # 6. Load the trained model and test inference
    xtts_inference.load_model(use_trained_model=True)
    
    # 7. Test voice cloning with the trained model
    print("\n🧪 Testing fine-tuned model capabilities...")
    xtts_inference.test_voice_cloning(reference_audios=tts_dataset['audio_path'].tolist()[:5])  # Use first 5 samples
    
    # 8. Benchmark performance
    benchmark_model_performance(xtts_inference, tts_dataset)
    
    print("\n🎉 PRODUCTION-READY XTTS v2 Model Training Complete!")
    print(f"   📊 Dataset: {len(tts_dataset)} samples")
    print(f"   🎤 Speakers: {tts_dataset['speaker_id'].nunique()}")
    print(f"   🌍 Languages: {list(tts_dataset['language'].unique())}")
    print(f"   💾 Model saved to: {xtts_inference.trained_model_path}")
    print(f"   🔧 Training logs: {trainer.training_output_path}")
    
    # 9. Show model usage instructions
    print("\n📋 How to use your trained model:")
    print("   1. The trained model is saved in:", xtts_inference.trained_model_path)
    print("   2. Load it using: inference.load_model(use_trained_model=True)")
    print("   3. Generate speech using the test_voice_cloning method")
    
else:
    print("\n⚠️ Training pipeline failed - check data availability or errors above.")
    print("   Falling back to base model testing...")
    
    # Fallback: Load base model and test
    if xtts_inference.load_model(use_trained_model=False):
        print("   Testing base XTTS v2 model...")
        xtts_inference.test_voice_cloning(reference_audios=tts_dataset['audio_path'].tolist()[:3])
        benchmark_model_performance(xtts_inference, tts_dataset)

In [ ]:
# 📋 IMPORTANT: Real XTTS v2 Training Requirements & Notes
print("📋 XTTS v2 Training Requirements:")
print("=" * 50)
print("🔧 HARDWARE REQUIREMENTS:")
print("   • Minimum: 12GB GPU RAM (Tesla T4, RTX 3060)")  
print("   • Recommended: 24GB+ GPU RAM (RTX 3090/4090, A100)")
print("   • Training time: 4-12+ hours depending on dataset size")
print("")
print("📊 DATA REQUIREMENTS:")
print("   • Minimum: 30+ minutes per speaker")
print("   • Recommended: 2+ hours per speaker")
print("   • Quality: Clean audio, accurate transcriptions")
print("")
print("⚠️  TRAINING CONSIDERATIONS:")
print("   • The training will actually fine-tune XTTS v2 model")
print("   • Progress will be shown with loss metrics")
print("   • Model checkpoints saved every 5000 steps")
print("   • Final model will be optimized for your specific voices")
print("")
print("🚀 Ready to start real training? Run the next cell!")
print("=" * 50)